# Lesson 1:  A first workflow

```{note}
Have you already installed everything you need?  If you're 
not sure, check the "Before you begin" section of the 
[introduction](../index.md). 
```

In this lesson. we'll construct a very simple workflow that simply
reads three numbers from its inputs and outputs their sum.  It's
nealry trivial, but this will demonstrate the essentials of working
with Tierkreis :
1. Defining a graph through its inputs and outputs
2. Constructing the computation
    - Using the inputs
    - Using simple nodes
    - Using built-in functionality
3. Checking what we've done with the visualizer
4. Running the workflow

Let's get started!

## Setting up the first graph

Workflows are defined by their *graphs* which described how the inputs produce the outputs.  The first step in defining the graph is to specify those inputs and outputs.  And part of that is knowing their names and their *types*.

Types are optional in Tierkreis, but since compile-time errors are cheap, and run-time errors are expensive, we strongly recommend using them everywhere.

```{note}
In order to keep a clear separation between the types used in the Tierkreis graph and the types already present in the Python
language we wrap the former with `TKR`.  in other words, the `TKR[A]` wrapper type indicates that an edge in the graph contains a value
of type `A`. More on this in the [core concepts](../tutorial/core_concepts.md#types)
```

In general a graph can have any number of inputs or outputs.  In this example, we have three inputs, so our first step is to create a python object to hold the inputs.  We always use a `NamedTuple` for this.  There's nothing special about the name `InParams`, it's just intended to be descriptive.

Since we have only one output, and we already know that the type is`TKR[float]`, we don't need to do anything for that.

In [ ]:
from typing import NamedTuple

from tierkreis.models import TKR


class InParams(NamedTuple):
    a: TKR[float]
    b: TKR[float]
    c: TKR[float]

Graphs are built using a `Graph`.  It needs to be instantiated with the types of its inputs and outputs.

In [ ]:
from tierkreis.builder import Graph

g = Graph(InParams, TKR[float])

## Building the Graph

Initially the graph doesn't do anything.  To change this we need to add some nodes to the graph.  There are several kinds of nodes we can have - for example Inputs and Outputs are nodes - but the most useful kind are called *Tasks*.  Roughly speaking a task is any kind of computation.  Most tasks are done by *Workers*, but for basic things, like adding numbers together, we can rely on *built-ins*.  Tierkreis has many built-ins [API docs](#tierkreis.builtins.main) but the only one we will need today is floating point addition, aka `add`.


In [ ]:
from tierkreis.builtins import add

Next we're going to construct a graph by calling the builder functions.
Each function adds a node to the graph; the arguments are the input edges to the node, and the return values are its output edges.  Notice that `inputs` is a special node created when we instantiated `g` and the names of the inputs are the ones we chose when defing the `InParams` class earlier.

In [ ]:
x = g.task(add(g.inputs.a, g.inputs.b))
y = g.task(add(x, g.inputs.c))

Finally, we have to say which edges of our graph will be the outputs.  In our example there's only one.
Once you call `finish_with_outputs` the graph can be run as a `Workflow`.
With this call you also indicate that the graph wont be changed later

In [ ]:
workflow = g.finish_with_outputs(y)

Congratulations - you build your first workflow!  But what does it look like?

## Using the visualizer

Tierkreis comes with an additional library to keep track of your workflows.  The main use is to observe a running workflow, but you can also use it to examine graphs that you are currently constructing.
```{info}
If you're running this from the tierkreis repository you need to set up the frontend once by runninig `just prod`.
```

The visualizer will run a local web application in the same process. To stop its execution you need to user `ctrl+c`. 

In [ ]:
%%script false --no-raise-error
from tierkreis_visualization.visualize_graph import visualize_graph

visualize_graph(g)

Opening the web interface at `localhost:8000` will show the landing page with the workflow overview.
![Landing Page](../_static/first_graph_overview.png)

After selecting the `tmp` workflow you will see the graph representation you just created.
![Graph](../_static/first_graph.png)

It shows the three input nodes `a,b,c`, the two task nodes `builtins.add` and an output with value `null` as the workflow hasn't run.
For the same reason all the nodes are depicted in white, which means they haven't been started yet.

To learn more about the visualizer see [this page](../tutorial/visualization.md)

## Running the workflow

The Rust-backed `Runtime` owns workflow state, input and output assets, and task execution. The default runtime discovers the installed `tkr-builtins` worker on `PATH`.


In [ ]:
from tierkreis import new_default

runtime = await new_default()


A newly-created in-memory runtime starts with no saved workflows or runs, so it does not require a workflow UUID or cleanup step.


In [ ]:
# The runtime is ready to accept workflows.


The runtime runs in the background while its context manager is active. Save the workflow, start a run, wait for attempt `0`, and then fetch its outputs.


In [ ]:
# Built-in tasks are executed by the runtime's in-memory executor.


As the penultimate step we need to provide the workflow inputs to run as a dictionary which we get from the input class..
If the inputs are not provided the workflow will encounter an error.

In [ ]:
inputs = InParams(0, 0.25, 0.5)._asdict()

The runtime returns a workflow ID when the graph is saved and a separate run ID for each execution.


In [ ]:
with runtime:
    workflow_id = await runtime.save_workflow("Hello World Graph", workflow)
    run_id = await runtime.start_new_run(workflow_id, inputs)
    await runtime.wait_for(run_id, 0)
    result = await runtime.get_outputs(run_id, 0)

print(result)
